# 02 - Integracion de datos a nivel de persona

Integra las fuentes de `data/processed/` en tablas agregadas a nivel `IDPERSONA`
(1 fila = 1 persona) y produce `data/features/dataset_personas_integrado.csv`.

**Regla principal:** nunca hacer JOIN directo entre tablas de detalle 1:N.
Cada fuente se agrega primero a nivel persona y luego se integra con LEFT JOIN
sobre una tabla base de personas.

**No se hace en este notebook:** preprocesamiento (ya esta en `data/processed/`),
clustering, embeddings, dashboard, ni seleccion final de variables para ML.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 60)
ENCODINGS = ("utf-8-sig", "cp1252", "latin-1")

## Funciones auxiliares reutilizables

In [ ]:
def leer_csv(nombre_archivo: str, **kwargs) -> pd.DataFrame:
    """Lee un CSV de data/processed probando distintas codificaciones."""
    ruta = PROCESSED_DIR / nombre_archivo
    ultimo_error = None
    for enc in ENCODINGS:
        try:
            df = pd.read_csv(ruta, encoding=enc, low_memory=False, **kwargs)
            print(f"Leido {nombre_archivo}: {df.shape[0]} filas x {df.shape[1]} columnas")
            return df
        except (UnicodeDecodeError, UnicodeError) as e:
            ultimo_error = e
    raise ultimo_error


def validar_llave(df: pd.DataFrame, llave: str, nombre: str) -> None:
    """Imprime diagnostico de una tabla de detalle antes de agregarla/unirla."""
    dups = df[llave].duplicated().sum()
    print(f"[{nombre}] llave={llave} | filas={len(df)} | "
          f"personas={df[llave].nunique(dropna=True)} | duplicados_llave={dups}")


def verificar_unicidad(df: pd.DataFrame, llave: str) -> bool:
    """True si `llave` identifica de forma unica cada fila de df."""
    return df[llave].is_unique


def validar_join(antes: pd.DataFrame, despues: pd.DataFrame, llave: str, nombre: str = "") -> None:
    """Compara filas/personas antes y despues de un LEFT JOIN sobre `llave`."""
    dup_despues = despues[llave].duplicated().sum()
    print(f"[JOIN {nombre}] filas: {len(antes)} -> {len(despues)} | "
          f"personas: {antes[llave].nunique()} -> {despues[llave].nunique()} | "
          f"duplicados_llave_despues={dup_despues}")
    if len(despues) != len(antes):
        print(f"  ADVERTENCIA: el numero de filas cambio tras el join ({nombre})")
    if dup_despues:
        print(f"  ADVERTENCIA: {llave} quedo duplicado tras el join ({nombre})")


def guardar_tabla(df: pd.DataFrame, nombre_archivo: str, llave: str = "IDPERSONA") -> Path:
    """Valida unicidad de `llave` y guarda la tabla en data/features/."""
    assert verificar_unicidad(df, llave), f"{nombre_archivo}: '{llave}' no es unico"
    ruta = FEATURES_DIR / nombre_archivo
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"Guardado {ruta.name}: {df.shape[0]} filas x {df.shape[1]} columnas")
    return ruta


def cobertura(numerador: pd.Series, denominador: pd.Series) -> pd.Series:
    """numerador / denominador elemento a elemento, evitando division por cero (-> NA)."""
    denom = denominador.replace(0, np.nan)
    return (numerador / denom).round(4)

## Carga de fuentes crudas (data/processed)

In [ ]:
titulaciones = leer_csv("reporte_titulaciones_educacion.csv").rename(columns={"IdPersona": "IDPERSONA"})
capacitaciones = leer_csv("capacitaciones_todas.csv")
certificados = leer_csv("certificados_todos.csv")
ponentes = leer_csv("ponentes_todos.csv")
carga_academica = leer_csv("carga_academica_disponible.csv")
carga_politecnica = leer_csv("carga_politecnica_disponible.csv")
catalogo_actividades = leer_csv("catalogo_actividades_carga.csv")
catalogo_idiomas = leer_csv("catalogo_idiomas.csv")
datos_personales = leer_csv("datos_personales_ultimos_5anios.csv")
experiencia_externa = leer_csv("experiencia_externa.csv")
heteroevaluacion = leer_csv("heteroevaluacion_disponible.csv")
idiomas_personas = leer_csv("idiomas_personas.csv")
mencion_honor = leer_csv("mencion_honor.csv")
proyecto_grado = leer_csv("proyecto_grado.csv")
proyectos_investigacion = leer_csv("proyectos_investigacion_disponible.csv")
proyectos_vinculacion = leer_csv("proyectos_vinculacion_disponible.csv")
publicaciones = leer_csv("publicaciones.csv")
historial_laboral_features = leer_csv("historial_laboral_features.csv")

## Relaciones ambiguas: no inventar joins

Antes de integrar, se revisaron 3 relaciones que no son evidentes y **no se
resuelven de forma automatica**:

| TABLA | LLAVE DISPONIBLE | PROBLEMA | DECISION RECOMENDADA |
|---|---|---|---|
| `catalogo_actividades_carga.csv` | `IDTIPOACTIVIDAD` (unico en el catalogo) | Solo ~7% de los `IDTIPOACTIVIDAD` de `carga_politecnica_disponible.csv` existen en el catalogo: son espacios de codigos distintos, no la misma clasificacion. | No unir. Se usa `carga_politecnica_disponible.csv` con sus propias columnas (`APLICA_T1/T2/T3`, horas) sin decodificar `IDTIPOACTIVIDAD`. |
| `catalogo_idiomas.csv` | `CODIGOSTR` (N/B/I/A) | No comparte llave con `IDIOMA`/`IDIDIOMA` de `idiomas_personas.csv`. Sus codigos (N/B/I/A) en realidad describen los niveles `NIVELLECTURA`/`NIVELESCRITURA`/`NIVELCONVERSACION`, no el idioma en si. | No se fusiona en `idiomas_persona.csv` (no aporta variable a nivel persona); queda documentado como diccionario de niveles, no de idiomas. |
| `proyecto_grado.csv` | `IDDIRECTOR` (sin columna `IDPERSONA` explicita) | Se verifico solapamiento de `IDDIRECTOR` contra la poblacion base (`historial_laboral_features` + `datos_personales`): ~70% (590/841) coincide. El resto puede ser personal fuera de la ventana de poblacion o datos externos. | Se asume `IDDIRECTOR == IDPERSONA` (unica llave disponible) y se renombra la columna. Los directores sin correspondencia quedaran como filas no encontradas en el LEFT JOIN final (no se inventa una persona). |

Ver detalle de la verificacion de `proyecto_grado` en la celda siguiente.

In [ ]:
# Verificacion de cobertura IDDIRECTOR vs poblacion (datos_personales_ultimos_5anios)
poblacion_ids = set(datos_personales["IDPERSONA"])
directores_ids = set(proyecto_grado["IDDIRECTOR"].dropna().unique())
coincidencias = directores_ids & poblacion_ids
print(f"IDDIRECTOR unicos: {len(directores_ids)} | coinciden con poblacion: {len(coincidencias)} "
      f"({len(coincidencias) / len(directores_ids):.1%})")

## Tabla base de personas

La poblacion se define **unicamente** por `IDPERSONA` en
`datos_personales_ultimos_5anios.csv` (personal contratado o vigente en los
ultimos 5 anios): 2213 personas. Todas las demas tablas (incluida
`historial_laboral_features.csv`) se integran con LEFT JOIN sobre esta base,
por lo que cualquier `IDPERSONA` de una fuente de detalle que no pertenezca a
esta poblacion (p. ej. directores de `proyecto_grado.csv` fuera de la ventana
de 5 anios) queda excluido del resultado final.

In [ ]:
validar_llave(datos_personales, "IDPERSONA", "datos_personales_ultimos_5anios")

personas = datos_personales.copy()
assert verificar_unicidad(personas, "IDPERSONA"), "IDPERSONA no es unico en datos_personales_ultimos_5anios"

guardar_tabla(personas, "personas.csv")
personas.head()

## Titulaciones (formacion academica)

Fuente: `reporte_titulaciones_educacion.csv`. Llave `IdPersona` (renombrada a
`IDPERSONA`). Puede tener varias titulaciones por persona; se agrega por
`IDPERSONA` y `IdTitulacion` para eliminar duplicados exactos antes de contar.

In [ ]:
validar_llave(titulaciones, "IDPERSONA", "reporte_titulaciones_educacion")

titulaciones_dedup = titulaciones.drop_duplicates(subset=["IDPERSONA", "IdTitulacion"])

niveles_dummy = pd.get_dummies(titulaciones_dedup["Nivel"], prefix="NUM_TIT")
niveles_dummy.columns = niveles_dummy.columns.str.replace(" ", "_")
niveles_por_persona = pd.concat([titulaciones_dedup["IDPERSONA"], niveles_dummy], axis=1).groupby("IDPERSONA").sum()

titulaciones_persona = (
    titulaciones_dedup.groupby("IDPERSONA")
    .agg(NUM_TITULACIONES=("IdTitulacion", "nunique"))
    .join(niveles_por_persona)
    .reset_index()
)
titulaciones_persona["TIENE_POSGRADO"] = (titulaciones_persona.get("NUM_TIT_CUARTO_NIVEL", 0) > 0).astype(int)

guardar_tabla(titulaciones_persona, "titulaciones_persona.csv")
titulaciones_persona.head()

## Capacitaciones

Fuente: `capacitaciones_todas.csv`. Se agrega por separado de certificaciones
y ponencias aunque compartan estructura, por ser conceptualmente distintas.

In [ ]:
validar_llave(capacitaciones, "IDPERSONA", "capacitaciones_todas")

capacitaciones_persona = capacitaciones.groupby("IDPERSONA").agg(
    NUM_CAPACITACIONES=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_CAPACITACIONES=("DURACION", "sum"),
    NUM_CAPACITACIONES_APROBACION=("TIPODESCRIPCION", lambda s: (s == "APROBACION").sum()),
    NUM_CAPACITACIONES_VIRTUAL=("TIPOMODALIDADDESCRIPCION", lambda s: (s == "VIRTUAL").sum()),
    NUM_CAPACITACIONES_PRESENCIAL=("TIPOMODALIDADDESCRIPCION", lambda s: (s == "PRESENCIAL").sum()),
).reset_index()

guardar_tabla(capacitaciones_persona, "capacitaciones_persona.csv")
capacitaciones_persona.head()

## Certificaciones

Fuente: `certificados_todos.csv`.

In [ ]:
validar_llave(certificados, "IDPERSONA", "certificados_todos")

certificaciones_persona = certificados.groupby("IDPERSONA").agg(
    NUM_CERTIFICACIONES=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_CERTIFICACIONES=("DURACION", "sum"),
).reset_index()

guardar_tabla(certificaciones_persona, "certificaciones_persona.csv")
certificaciones_persona.head()

## Ponencias

Fuente: `ponentes_todos.csv`.

In [ ]:
validar_llave(ponentes, "IDPERSONA", "ponentes_todos")

ponencias_persona = ponentes.groupby("IDPERSONA").agg(
    NUM_PONENCIAS=("IDCAPACITACION", "nunique"),
    SUMA_DURACION_PONENCIAS=("DURACION", "sum"),
).reset_index()

guardar_tabla(ponencias_persona, "ponencias_persona.csv")
ponencias_persona.head()

## Docencia (carga academica)

Fuente: `carga_academica_disponible.csv`. Una fila por curso/paralelo/periodo
asignado a un docente; se agrega por `IDPERSONA`.

In [ ]:
validar_llave(carga_academica, "IDPERSONA", "carga_academica_disponible")

docencia_persona = carga_academica.groupby("IDPERSONA").agg(
    NUM_CURSOS=("IDCURSO", "nunique"),
    NUM_PERIODOS_DOCENCIA=("IDPERIODO", "nunique"),
    NUM_ASIGNACIONES_DOCENCIA=("IDCPLCAMBIOSCURSO", "nunique"),
    TOTAL_HORAS_DOCENCIA=("TOTALHORAS", "sum"),
    TOTAL_ESTUDIANTES=("NUMREGISTRADOS", "sum"),
    PROMEDIO_ESTUDIANTES_POR_CURSO=("NUMREGISTRADOS", "mean"),
).reset_index()
docencia_persona["PROMEDIO_ESTUDIANTES_POR_CURSO"] = docencia_persona["PROMEDIO_ESTUDIANTES_POR_CURSO"].round(2)

guardar_tabla(docencia_persona, "docencia_persona.csv")
docencia_persona.head()

## Carga politecnica

Fuente: `carga_politecnica_disponible.csv`. Ya trae `APLICA_T1/T2/T3`,
`NUM_TERMINOS` y `TERMINOS_ACTIVOS` por actividad (no se recalculan). Se
agregan a nivel persona sumando/contando esas columnas existentes.
`NUMHORASCP` se usa como horas de carga politecnica (mismo orden de magnitud
que las cargas reportadas; `NUMHORAS` es una columna auxiliar mas pequeÃ±a).

In [ ]:
validar_llave(carga_politecnica, "IDPERSONA", "carga_politecnica_disponible")

carga_politecnica_persona = carga_politecnica.groupby("IDPERSONA").agg(
    NUM_ACTIVIDADES_POLITECNICAS=("IDACTIVIDAD", "nunique"),
    NUM_ACTIVIDADES_T1=("APLICA_T1", "sum"),
    NUM_ACTIVIDADES_T2=("APLICA_T2", "sum"),
    NUM_ACTIVIDADES_T3=("APLICA_T3", "sum"),
    TOTAL_HORAS_POLITECNICAS=("NUMHORASCP", "sum"),
).reset_index()

guardar_tabla(carga_politecnica_persona, "carga_politecnica_persona.csv")
carga_politecnica_persona.head()

## Experiencia externa

Fuente: `experiencia_externa.csv`. Se usan las columnas ya decodificadas
`CATEXPERIENCIA_DESC` y `ROLACADEMICO_DESC`.

In [ ]:
validar_llave(experiencia_externa, "IDPERSONA", "experiencia_externa")

experiencia_externa_persona = experiencia_externa.groupby("IDPERSONA").agg(
    NUM_EXPERIENCIAS_EXTERNAS=("IDHISTORIALABORAL", "nunique"),
    NUM_EXPERIENCIAS_ACADEMICAS=("CATEXPERIENCIA_DESC", lambda s: (s == "ACADEMICA").sum()),
    NUM_EXPERIENCIAS_ADMINISTRATIVAS=("CATEXPERIENCIA_DESC", lambda s: (s == "ADMINISTRATIVA").sum()),
    NUM_EXPERIENCIAS_POR_CLASIFICAR=("CATEXPERIENCIA_DESC", lambda s: (s == "POR CLASIFICAR").sum()),
    NUM_EXPERIENCIAS_ROL_PROFESOR=("ROLACADEMICO_DESC", lambda s: (s == "PROFESOR").sum()),
).reset_index()

guardar_tabla(experiencia_externa_persona, "experiencia_externa_persona.csv")
experiencia_externa_persona.head()

## Idiomas

Fuente: `idiomas_personas.csv`. No se convierte `NIVELMCER` (A1-C2) a numero
todavia (queda para la etapa de preparacion para clustering). `catalogo_idiomas.csv`
no se fusiona aqui (ver seccion de relaciones ambiguas).

In [ ]:
validar_llave(idiomas_personas, "IDPERSONA", "idiomas_personas")

idiomas_persona = idiomas_personas.groupby("IDPERSONA").agg(
    NUM_IDIOMAS=("IDIDIOMA", "nunique"),
    NUM_IDIOMAS_NO_NATIVOS=("LENGUANATIVA", lambda s: (s == 0).sum()),
    NUM_IDIOMAS_CON_NIVELMCER=("NIVELMCER", "count"),
).reset_index()

guardar_tabla(idiomas_persona, "idiomas_persona.csv")
idiomas_persona.head()

## Heteroevaluacion docente

Fuente: `heteroevaluacion_disponible.csv`. Una fila por curso/periodo
evaluado; se agrega **primero por `IDPERSONA`** (no se une fila a fila con
`carga_academica_disponible.csv`).

In [ ]:
validar_llave(heteroevaluacion, "IDPERSONA", "heteroevaluacion_disponible")

evaluacion_persona = heteroevaluacion.groupby("IDPERSONA").agg(
    NUM_EVALUACIONES=("IDCURSO", "count"),
    PROMEDIO_HETEROEVALUACION=("PROMEDIO", "mean"),
    TOTAL_REGISTRADOS=("REGISTRADOS", "sum"),
    TOTAL_EVALUADOS=("EVALUADOS", "sum"),
).reset_index()
evaluacion_persona["PROMEDIO_HETEROEVALUACION"] = evaluacion_persona["PROMEDIO_HETEROEVALUACION"].round(2)
evaluacion_persona["COBERTURA_EVALUACION"] = cobertura(
    evaluacion_persona["TOTAL_EVALUADOS"], evaluacion_persona["TOTAL_REGISTRADOS"]
)

guardar_tabla(evaluacion_persona, "evaluacion_persona.csv")
evaluacion_persona.head()

## Reconocimientos (menciones de honor)

Fuente: `mencion_honor.csv`.

In [ ]:
validar_llave(mencion_honor, "IDPERSONA", "mencion_honor")

reconocimientos_persona = mencion_honor.groupby("IDPERSONA").agg(
    NUM_RECONOCIMIENTOS=("IDMENCIONHONOR", "nunique"),
    NUM_TIPOS_RECONOCIMIENTO_DISTINTOS=("TIPO", "nunique"),
).reset_index()

guardar_tabla(reconocimientos_persona, "reconocimientos_persona.csv")
reconocimientos_persona.head()

## Direccion de trabajos de titulacion (proyecto de grado)

Fuente: `proyecto_grado.csv`. Solo trae `IDDIRECTOR` (ver decision documentada
arriba): se renombra a `IDPERSONA` antes de agregar.

In [ ]:
proyecto_grado_persona_src = proyecto_grado.rename(columns={"IDDIRECTOR": "IDPERSONA"}).dropna(subset=["IDPERSONA"])
proyecto_grado_persona_src["IDPERSONA"] = proyecto_grado_persona_src["IDPERSONA"].astype("int64")
validar_llave(proyecto_grado_persona_src, "IDPERSONA", "proyecto_grado (IDDIRECTOR->IDPERSONA)")

proyecto_grado_persona = proyecto_grado_persona_src.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_GRADO_DIRIGIDOS=("NOMBRETRABAJOTITULACION", "count"),
    NUM_PROGRAMAS_TITULACION_DISTINTOS=("NOMBREPROGRAMA", "nunique"),
).reset_index()

guardar_tabla(proyecto_grado_persona, "proyecto_grado_persona.csv")
proyecto_grado_persona.head()

## Proyectos de investigacion

Fuente: `proyectos_investigacion_disponible.csv`.

In [ ]:
validar_llave(proyectos_investigacion, "IDPERSONA", "proyectos_investigacion_disponible")

investigacion_persona = proyectos_investigacion.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_INVESTIGACION=("IDPROYECTOINVESTIGACION", "nunique"),
    NUM_PROYECTOS_ACTIVOS=("ESTADO_PROYECTO", lambda s: s.str.contains("EJECUCI", na=False).sum()),
    NUM_PROYECTOS_FINALIZADOS=("ESTADO_PROYECTO", lambda s: s.isin(["FINALIZADO", "CERRADO"]).sum()),
    NUM_PROYECTOS_COMO_DIRECTOR=("ROLPROYECTO", lambda s: (s == "DIRECTOR").sum()),
    NUM_PROYECTOS_COMO_CODIRECTOR=("ROLPROYECTO", lambda s: (s == "CO-DIRECTOR").sum()),
    NUM_PROYECTOS_COMO_PARTICIPANTE=("ROLPROYECTO", lambda s: (s == "PARTICIPANTE").sum()),
).reset_index()

guardar_tabla(investigacion_persona, "investigacion_persona.csv")
investigacion_persona.head()

## Proyectos de vinculacion

Fuente: `proyectos_vinculacion_disponible.csv`.

In [ ]:
validar_llave(proyectos_vinculacion, "IDPERSONA", "proyectos_vinculacion_disponible")

vinculacion_persona = proyectos_vinculacion.groupby("IDPERSONA").agg(
    NUM_PROYECTOS_VINCULACION=("IDPROYECTOINVESTIGACION", "nunique"),
    NUM_VINCULACION_TUTOR=("ROLPROYECTO", lambda s: (s == "TUTOR").sum()),
    NUM_VINCULACION_DIRECTOR_PROYECTO=("ROLPROYECTO", lambda s: (s == "DIRECTOR DE PROYECTO").sum()),
    NUM_VINCULACION_DIRECTOR_PROGRAMA=("ROLPROYECTO", lambda s: (s == "DIRECTOR DE PROGRAMA").sum()),
).reset_index()

guardar_tabla(vinculacion_persona, "vinculacion_persona.csv")
vinculacion_persona.head()

## Publicaciones

Fuente: `publicaciones.csv`. Se usan directamente las columnas ya
codificadas (`REVISTAINDEXADA`, `REVISIONPARES`, `CUARTIL`, `CUARTILCITESCORE`).

In [ ]:
validar_llave(publicaciones, "IDPERSONA", "publicaciones")

publicaciones_persona = publicaciones.groupby("IDPERSONA").agg(
    NUM_PUBLICACIONES=("IDPUBLICACIONPERSONA", "nunique"),
    NUM_PUBLICACIONES_INDEXADAS=("REVISTAINDEXADA", lambda s: (s == 1).sum()),
    NUM_PUBLICACIONES_REVISION_PARES=("REVISIONPARES", lambda s: (s == 1).sum()),
    NUM_PUBLICACIONES_Q1=("CUARTIL", lambda s: (s == "Q1").sum()),
    NUM_PUBLICACIONES_Q2=("CUARTIL", lambda s: (s == "Q2").sum()),
).reset_index()

guardar_tabla(publicaciones_persona, "publicaciones_persona.csv")
publicaciones_persona.head()

## Historial laboral (ya agregado)

`historial_laboral_features.csv` ya fue construido en `10_historial_laboral_personas.ipynb`
a nivel de persona. Aqui solo se valida su unicidad; **no se vuelve a agregar**
ni se crea `historial_laboral_persona.csv`.

In [ ]:
validar_llave(historial_laboral_features, "IDPERSONA", "historial_laboral_features")
assert verificar_unicidad(historial_laboral_features, "IDPERSONA"), "IDPERSONA no es unico en historial_laboral_features"
print("historial_laboral_features OK: IDPERSONA es unico")

## Integracion final

LEFT JOIN de todas las tablas agregadas sobre la tabla base `personas`. Se
valida numero de filas/personas y duplicados de `IDPERSONA` despues de cada
join.

In [ ]:
tablas_a_integrar = [
    ("titulaciones_persona", titulaciones_persona),
    ("capacitaciones_persona", capacitaciones_persona),
    ("certificaciones_persona", certificaciones_persona),
    ("ponencias_persona", ponencias_persona),
    ("docencia_persona", docencia_persona),
    ("carga_politecnica_persona", carga_politecnica_persona),
    ("experiencia_externa_persona", experiencia_externa_persona),
    ("idiomas_persona", idiomas_persona),
    ("evaluacion_persona", evaluacion_persona),
    ("reconocimientos_persona", reconocimientos_persona),
    ("proyecto_grado_persona", proyecto_grado_persona),
    ("investigacion_persona", investigacion_persona),
    ("vinculacion_persona", vinculacion_persona),
    ("publicaciones_persona", publicaciones_persona),
    ("historial_laboral_features", historial_laboral_features),
]

dataset_personas_integrado = personas.copy()
for nombre, tabla in tablas_a_integrar:
    assert verificar_unicidad(tabla, "IDPERSONA"), f"{nombre}: IDPERSONA no es unico, no se puede unir"
    antes = dataset_personas_integrado.copy()
    dataset_personas_integrado = dataset_personas_integrado.merge(tabla, on="IDPERSONA", how="left")
    validar_join(antes, dataset_personas_integrado, "IDPERSONA", nombre)

# Contadores (NUM_*) se rellenan con 0 cuando la persona no aparece en la fuente
columnas_conteo = [c for c in dataset_personas_integrado.columns if c.startswith(("NUM_", "TOTAL_", "SUMA_"))]
dataset_personas_integrado[columnas_conteo] = dataset_personas_integrado[columnas_conteo].fillna(0)

## Validacion final

In [ ]:
assert dataset_personas_integrado["IDPERSONA"].is_unique, "IDPERSONA no es unico en el dataset final"

print(f"Personas totales: {dataset_personas_integrado['IDPERSONA'].nunique()}")
print(f"Columnas totales: {dataset_personas_integrado.shape[1]}")
print(f"Duplicados de IDPERSONA: {dataset_personas_integrado['IDPERSONA'].duplicated().sum()}")

nulos_pct = (dataset_personas_integrado.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nPorcentaje de nulos por columna (top 20):")
print(nulos_pct.head(20))

In [ ]:
guardar_tabla(dataset_personas_integrado, "dataset_personas_integrado.csv")
dataset_personas_integrado.head()

## Visualizacion del dataset integrado

Graficos exploratorios sobre `dataset_personas_integrado.csv` (2213 personas):
cobertura de cada fuente, calidad de datos (nulos), demografia, relacion
laboral y correlacion entre indicadores agregados. Objetivo: entender la data
integrada, no preparar variables para clustering.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Paleta (misma logica que las demas graficas del proyecto): un solo tono para
# magnitudes de una sola serie, paleta categorica de orden fijo para grupos.
COLOR_BLUE, COLOR_ORANGE, COLOR_AQUA, COLOR_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
PALETA_CATEGORICA = [COLOR_BLUE, COLOR_ORANGE, COLOR_AQUA, COLOR_YELLOW, "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10,
})


def _estilizar_ejes(ax):
    """Quita bordes sobrantes y deja los ejes en tono recesivo."""
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.set_axisbelow(True)


def grafico_barh_pct(labels, valores, titulo, xlabel="% de personas", color=COLOR_BLUE):
    """Barras horizontales para un solo indicador de magnitud (ej. cobertura, % nulos)."""
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.35 * len(labels))))
    y = range(len(labels))
    ax.barh(y, valores, color=color, height=0.6, zorder=3)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    for i, v in enumerate(valores):
        ax.text(v + max(valores) * 0.015, i, f"{v:.0f}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()


def grafico_hist(serie, titulo, xlabel, color=COLOR_BLUE, bins=25):
    """Histograma simple para una variable numerica continua."""
    fig, ax = plt.subplots(figsize=(7, 4))
    datos = serie.dropna()
    ax.hist(datos, bins=bins, color=color, edgecolor="#fcfcfb", linewidth=0.6, zorder=3)
    ax.set_xlabel(xlabel); ax.set_ylabel("N.º de personas")
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    plt.tight_layout()
    plt.show()


def grafico_barras_categoria(serie, titulo, orden=None, colores=PALETA_CATEGORICA):
    """Barras verticales de conteo para una variable categorica (orden de color fijo)."""
    conteo = serie.dropna().value_counts()
    if orden is not None:
        conteo = conteo.reindex(orden).dropna()
    fig, ax = plt.subplots(figsize=(max(4, 1.2 * len(conteo)), 4))
    ax.bar(conteo.index.astype(str), conteo.values, color=colores[: len(conteo)], zorder=3)
    ax.set_ylabel("N.º de personas")
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    for i, v in enumerate(conteo.values):
        ax.text(i, v + max(conteo.values) * 0.015, f"{v}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()

### Cobertura de personas por fuente de datos

Porcentaje de las 2213 personas de la poblacion que tienen al menos un
registro en cada fuente. Muestra que fuentes son casi universales (historial
laboral, formacion) y cuales cubren solo un subconjunto (investigacion,
vinculacion, publicaciones, ponencias), algo clave para interpretar despues
los ceros de las columnas `NUM_*` (no siempre significan "no aplica").

In [ ]:
fuentes_cobertura = {
    "Historial laboral": "N_REGISTROS_HISTORIAL",
    "Formacion (titulaciones)": "NUM_TITULACIONES",
    "Capacitaciones": "NUM_CAPACITACIONES",
    "Docencia": "NUM_CURSOS",
    "Heteroevaluacion": "NUM_EVALUACIONES",
    "Carga politecnica": "NUM_ACTIVIDADES_POLITECNICAS",
    "Experiencia externa": "NUM_EXPERIENCIAS_EXTERNAS",
    "Idiomas": "NUM_IDIOMAS",
    "Certificaciones": "NUM_CERTIFICACIONES",
    "Reconocimientos": "NUM_RECONOCIMIENTOS",
    "Investigacion": "NUM_PROYECTOS_INVESTIGACION",
    "Publicaciones": "NUM_PUBLICACIONES",
    "Ponencias": "NUM_PONENCIAS",
    "Vinculacion": "NUM_PROYECTOS_VINCULACION",
    "Direccion de tesis": "NUM_PROYECTOS_GRADO_DIRIGIDOS",
}
cobertura_pct = pd.Series({
    nombre: (dataset_personas_integrado[col].fillna(0) > 0).mean() * 100
    for nombre, col in fuentes_cobertura.items()
}).sort_values(ascending=False)

grafico_barh_pct(cobertura_pct.index.tolist(), cobertura_pct.values, "Cobertura de personas por fuente de datos")

### Columnas con mayor proporcion de nulos

Complemento visual de la tabla de nulos calculada en la validacion final.
Los nulos restantes son sobre todo demograficos (`datos_personales`) y de
dedicacion docente (aplica solo a quienes son docentes).

In [ ]:
grafico_barh_pct(
    nulos_pct.head(15).index.tolist(), nulos_pct.head(15).values,
    "Top 15 columnas con mayor % de nulos", xlabel="% de nulos", color=COLOR_ORANGE,
)

### Demografia

Distribucion de edad y sexo de la poblacion.

In [ ]:
grafico_hist(dataset_personas_integrado["EDAD"], "Distribucion de edad", "Edad (anios)")
grafico_barras_categoria(dataset_personas_integrado["SEXO"], "Personas por sexo", orden=["M", "F"])

### Relacion laboral

Antiguedad efectiva (anios de vinculacion real, sin contar brechas), tipo de
empleado actual y dedicacion docente (`historial_laboral_features`).

In [ ]:
grafico_hist(
    dataset_personas_integrado["ANTIGUEDAD_EFECTIVA_ANIOS"],
    "Distribucion de antiguedad efectiva", "Antiguedad efectiva (anios)",
)
grafico_barras_categoria(
    dataset_personas_integrado["TIPOEMPLEADO_ACTUAL_DESC"], "Personas por tipo de empleado actual",
)
grafico_barras_categoria(
    dataset_personas_integrado["DEDICACION_DOCENTE_ACTUAL"], "Docentes por dedicacion actual",
    orden=["Tiempo Completo", "Tiempo Parcial", "Medio Tiempo", "No Aplica"],
)

### Correlacion entre indicadores agregados

Correlacion de Pearson entre variables numericas clave de distintas fuentes
(demografia, historial laboral, docencia, formacion, produccion academica).
Solo para explorar relaciones entre fuentes ya integradas; no implica
seleccion de variables para clustering.

In [ ]:
columnas_correlacion = [
    "EDAD", "ANTIGUEDAD_EFECTIVA_ANIOS", "N_CONTRATOS_TOTAL",
    "NUM_TITULACIONES", "NUM_CAPACITACIONES", "NUM_CURSOS", "TOTAL_HORAS_DOCENCIA",
    "PROMEDIO_HETEROEVALUACION", "NUM_IDIOMAS", "TOTAL_HORAS_POLITECNICAS",
    "NUM_PUBLICACIONES", "NUM_PROYECTOS_INVESTIGACION", "NUM_PROYECTOS_VINCULACION",
]
corr = dataset_personas_integrado[columnas_correlacion].corr()

cmap_diverg = LinearSegmentedColormap.from_list("blue_gray_red", ["#e34948", "#f0efec", "#2a78d6"])
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap=cmap_diverg, vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas_correlacion))); ax.set_xticklabels(columnas_correlacion, rotation=45, ha="right")
ax.set_yticks(range(len(columnas_correlacion))); ax.set_yticklabels(columnas_correlacion)
for i in range(len(columnas_correlacion)):
    for j in range(len(columnas_correlacion)):
        valor = corr.iloc[i, j]
        color_texto = "#ffffff" if abs(valor) > 0.6 else INK
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7, color=color_texto)
ax.set_title("Correlacion entre indicadores agregados por persona", loc="left", fontsize=12, pad=12)
fig.colorbar(im, ax=ax, shrink=0.8, label="Correlacion de Pearson")
plt.tight_layout()
plt.show()